# Training curves (MLflow)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gen_cats.evaluation.experiment_artifacts import (
    load_metric_history,
    load_mlflow_runs,
    match_mlflow_to_fid_cell,
    monitor_metric_for_model,
    pick_mlflow_run,
)
from gen_cats.evaluation.report_analysis import (
    best_runs_table,
    ensure_plots_dir,
    load_fid_scores,
    write_stats_csv,
)

PLOTS = ensure_plots_dir("results")
ml = load_mlflow_runs()
best_rows = {r["model"]: r for r in best_runs_table(load_fid_scores())}
best_rows

In [ ]:
curve_runs = []
curve_models = [m for m in best_rows if best_rows[m].get("hyperparameters")]
n_models = len(curve_models)
fig, axes = plt.subplots(1, n_models, figsize=(3.6 * n_models, 3.2))
if n_models == 1:
    axes = [axes]
for ax, model in zip(axes, curve_models, strict=True):
    row = best_rows[model]
    metric = monitor_metric_for_model(model)
    matched = match_mlflow_to_fid_cell(ml, model, row["hyperparameters"])
    if matched.empty:
        ax.axis("off")
        continue
    for seed in sorted(matched["seed"].dropna().unique()):
        run = pick_mlflow_run(matched[matched["seed"] == seed])
        if run is None:
            continue
        hist = load_metric_history(run["run_uuid"], metric)
        if hist.empty:
            continue
        ax.plot(hist["epoch"], hist["value"], label=f"seed {int(seed)}")
        curve_runs.append(
            {
                "model": model,
                "seed": int(seed),
                "run_uuid": run["run_uuid"],
                "status": run.get("status"),
                "metric": metric,
                "n_points": len(hist),
            }
        )
    ax.set_title(row["label"])
    if ax.lines:
        ax.legend(fontsize=7)
fig.tight_layout()
out = PLOTS / "training_curves_best_fid.png"
fig.savefig(out, dpi=180, bbox_inches="tight")
plt.close(fig)
write_stats_csv(pd.DataFrame(curve_runs), "training_curve_runs.csv")
out